In [1]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/ammara-asif/fyp.git"   # <-- EDIT ONCE
BRANCH = "stage1-nested"   # the branch with the code this notebook needs (set back to "main" after merging)

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", url, CODE], check=True)
# fetch + checkout + pull, so a clone left on another branch earlier in the session switches over
subprocess.run(["git", "-C", CODE, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", CODE, "pull", "-q", "origin", BRANCH], check=True)
print(BRANCH, subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
_MINE = {"fyp_infra", "stage1_runner", "stage1_analysis", "stage1_drift_diagnostic",
         "stage1_structural", "nested_builder"}
for _m in list(sys.modules):
    if _m in _MINE or _m == "src" or _m.startswith("src."):   # src.* = the team's nested code
        sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop for training: a CPU run is ~12x slower and its timings are useless for budgeting.
REQUIRE_GPU = False   # analysis only: CPU runtime is fine
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

Mounted at /content/drive
stage1-nested 9a30690 updated



/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ammara-asif125 (kashaf-saleem09-institute-of-business-administration) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


{'root': '/content/drive/MyDrive/fyp_nested_cl', 'data': '/content/drive/MyDrive/fyp_nested_cl/data', 'results': '/content/drive/MyDrive/fyp_nested_cl/results', 'logs': '/content/drive/MyDrive/fyp_nested_cl/logs', 'checkpoints': '/content/drive/MyDrive/fyp_nested_cl/checkpoints'}
GPU: Tesla T4


In [2]:
import stage1_analysis as sa
import stage1_structural as st
STAGES = ["stage1_control", "stage1_nested_control", "stage1_epochs_per_task"]   # old control, Flat/B/D via the nested path, steps-per-task test
TRAIN_RUNS = sa.list_runs(paths, stage=STAGES)
for r in TRAIN_RUNS:
    c = r["config"]; print(r["run_id"], sr.arm_label(c), "lr", c["lr"], "seed", c["seed"])
SCFGS = st.structural_configs(TRAIN_RUNS)

06a0c117c983 flat_nl_ep2 lr 0.1 seed 1
07a97dd12ec3 flat_nl lr 0.1 seed 0
1803d732791d llrd_fast_deep lr 0.6 seed 1
1a6e0b7da780 nested_a lr 0.5 seed 1
32e5df8b8886 flat lr 0.3 seed 2
3faea1c21912 nested_b lr 0.6 seed 3
427e05b19ed5 nested_b_ep2 lr 0.6 seed 1
48523f3dfeea nested_b lr 0.6 seed 1
4b2bd0082f5c flat lr 0.1 seed 0
585d82e26d13 nested_b_ep3 lr 0.6 seed 2
59e84252cd1c flat_nl_ep3 lr 0.1 seed 1
6459881d31c2 nested_b_ep2 lr 0.6 seed 0
6fa1e584d18e nested_b_ep3 lr 0.6 seed 0
78e133183087 flat_nl lr 0.1 seed 1
7b3073002436 flat lr 0.3 seed 0
82d25ec4d308 nested_c_cw0.1 lr 0.1 seed 2
82f66588c5f1 nested_a lr 0.5 seed 0
87e47e2a7fd9 flat_nl_ep3 lr 0.1 seed 0
892486433e6d flat lr 0.1 seed 1
909b2256d515 nested_a lr 0.5 seed 4
97e56eaf8f25 flat_nl lr 0.1 seed 3
9940a100c076 flat_nl_ep3 lr 0.1 seed 2
9d5d9fcc4194 llrd_fast_deep lr 0.6 seed 0
9da44b28d332 nested_a lr 0.5 seed 2
a1e14e58bff7 flat_nl lr 0.1 seed 4
b09e3cc29bda flat_nl lr 0.1 seed 2
b2316f2cb5f4 nested_d_lam0.01 lr 0.7 se

In [3]:
out = run_sweep(SCFGS, st.structural_run, paths)

[1/45] 68f590e39ccf already finished, skipping
[2/45] 1f493c365153 already finished, skipping
[3/45] b7c5dedbcd65 already finished, skipping
[4/45] 3ebb2d30623d already finished, skipping
[5/45] ccacb1ec8e62 already finished, skipping
[6/45] 61d6cc0ae742 running
  checkpoint 1/20
  checkpoint 2/20
  checkpoint 3/20
  checkpoint 4/20
  checkpoint 5/20
  checkpoint 6/20
  checkpoint 7/20
  checkpoint 8/20
  checkpoint 9/20
  checkpoint 10/20
  checkpoint 11/20
  checkpoint 12/20
  checkpoint 13/20
  checkpoint 14/20
  checkpoint 15/20
  checkpoint 16/20
  checkpoint 17/20
  checkpoint 18/20
  checkpoint 19/20
  checkpoint 20/20
[7/45] 1eb6a14ad686 already finished, skipping
[8/45] cdcd2d2a4a6a already finished, skipping
[9/45] 8c38c8e243b4 already finished, skipping
[10/45] 3281a766b03c already finished, skipping
[11/45] 32406c1cb700 already finished, skipping
[12/45] ecd5aa8b0b2b already finished, skipping
[13/45] 8a25c51e196c already finished, skipping
[14/45] 5a5aa34fa529 already fini

In [4]:
import numpy as np, pandas as pd
rows = []
for a in SCFGS:
    r = load_result(a, paths)
    if r is None: continue
    m = r["metrics"]; c = m["train_config"]
    row = {"arm": sr.arm_label(c), "lr": c["lr"], "seed": c["seed"]}
    for l, name in enumerate(m["level_modules"]):
        row[f"cka_to_final/{name}"] = m["cka_to_final_avg"][l]
        row[f"cka_consecutive/{name}"] = m["cka_consecutive_avg"][l]
    # group names differ between the two model classes (conv1+bn1+layer1 vs stem+layer1), so label by position
    n_g = len(m["grad_cos_avg"])
    for g, v in enumerate(m["grad_cos_avg"]):
        row["grad_cos/" + ("head" if g == n_g - 1 else f"group{g}")] = v
    rows.append(row)
sdf = pd.DataFrame(rows)
display(sdf.drop(columns="seed").groupby(["arm", "lr"]).agg(["mean", "std"]).T.round(3))

arm                           flat        flat_nl flat_nl_ep2 flat_nl_ep3  \
lr                             0.1    0.3     0.1         0.1         0.1   
cka_to_final/layer1    mean  0.748  0.745   0.757       0.736       0.736   
                       std   0.038  0.032   0.043       0.076       0.023   
cka_consecutive/layer1 mean  0.793  0.765   0.810       0.803       0.777   
                       std   0.030  0.021   0.026       0.040       0.017   
cka_to_final/layer3    mean  0.589  0.614   0.583       0.514       0.519   
                       std   0.054  0.027   0.031       0.056       0.068   
cka_consecutive/layer3 mean  0.555  0.584   0.569       0.558       0.528   
                       std   0.020  0.012   0.031       0.085       0.032   
cka_to_final/layer4    mean  0.407  0.387   0.407       0.433       0.414   
                       std   0.036  0.011   0.031       0.041       0.039   
cka_consecutive/layer4 mean  0.290  0.278   0.301       0.389       0.342   
                       std   0.020  0.029   0.040       0.064       0.047   
grad_cos/group0        mean  0.398  0.421   0.372       0.396       0.343   
                       std   0.113  0.031   0.101       0.060       0.120   
grad_cos/group1        mean  0.388  0.434   0.364       0.349       0.345   
                       std   0.101  0.057   0.084       0.048       0.104   
grad_cos/group2        mean  0.265  0.316   0.243       0.151       0.186   
                       std   0.077  0.061   0.058       0.042       0.049   
grad_cos/head          mean  0.207  0.258   0.194       0.162       0.178   
                       std   0.036  0.053   0.039       0.030       0.034   

arm                         llrd_fast_deep llrd_slow_deep nested_a nested_b  \
lr                                     0.6            0.5      0.5      0.6   
cka_to_final/layer1    mean          0.730          0.739    0.785    0.773   
                       std           0.052          0.061    0.040    0.016   
cka_consecutive/layer1 mean          0.833          0.798    0.865    0.828   
                       std           0.021          0.016    0.018    0.015   
cka_to_final/layer3    mean          0.565          0.537    0.627    0.615   
                       std           0.058          0.074    0.029    0.044   
cka_consecutive/layer3 mean          0.578          0.549    0.643    0.599   
                       std           0.015          0.027    0.019    0.038   
cka_to_final/layer4    mean          0.388          0.392    0.408    0.397   
                       std           0.049          0.045    0.032    0.036   
cka_consecutive/layer4 mean          0.274          0.317    0.307    0.287   
                       std           0.027          0.026    0.033    0.026   
grad_cos/group0        mean          0.368          0.330    0.342    0.381   
                       std           0.059          0.150    0.051    0.134   
grad_cos/group1        mean          0.362          0.329    0.341    0.352   
                       std           0.074          0.126    0.025    0.119   
grad_cos/group2        mean          0.268          0.236    0.229    0.271   
                       std           0.048          0.090    0.036    0.061   
grad_cos/head          mean          0.203          0.203    0.180    0.212   
                       std           0.032          0.069    0.014    0.046   

arm                         nested_b_ep2 nested_b_ep3 nested_c_cw0.1  \
lr                                   0.6          0.6            0.1   
cka_to_final/layer1    mean        0.783        0.771          0.771   
                       std         0.054        0.023          0.028   
cka_consecutive/layer1 mean        0.857        0.847          0.813   
                       std         0.042        0.023          0.004   
cka_to_final/layer3    mean        0.536        0.531          0.587   
                       std         0.097        0.093         